In [ ]:
# === CHALLENGE 2 (TNF-alpha) : PRODUCTION SETUP ===
# Builds the CROPPED target. The smoke test measured 43.3 s per iteration on the full
# two-chain construct (304 + 70 residues) - 7.2x Challenge 1's rate for a 1.7x length
# increase, so the cost scales nearer L^3.7 than L^2. At that rate the whole GPU budget
# buys about six designs, which is not worth running.
#
# Cropping to a 22 A sphere around the switch site keeps all 25 site residues, both
# switch anchors (K166 of chain A, R108 of chain B) and - measured against the intact
# trimer - introduces ZERO artificially exposed surface at the site, fewer than the
# uncropped two-chain construct, because cropping also removes residues that chain C
# had been covering.
import glob, json, os, shutil, subprocess, sys, time
t0 = time.time()
OFF = 76
RADIUS = 22.0


def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("FAILED:", cmd)
        print((r.stdout + r.stderr)[-1000:])
        if check:
            raise SystemExit("setup failed")
    return r


print("python", sys.version.split()[0], flush=True)
print(subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip(), flush=True)

os.makedirs("/kaggle/working/params", exist_ok=True)
src = glob.glob("/kaggle/input/**/params/params_model_*.npz", recursive=True)
if src:
    print("reusing %d AF2 param files from the attached input" % len(src), flush=True)
    for f in src:
        dst = "/kaggle/working/params/" + os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy(f, dst)
else:
    print("downloading AF2 params (~3.5 GB) ...", flush=True)
    sh("wget -q -O /kaggle/working/af2.tar "
       "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar")
    sh("tar -xf /kaggle/working/af2.tar -C /kaggle/working/params")
    sh("rm -f /kaggle/working/af2.tar")
print("AF2 params present:", len(glob.glob("/kaggle/working/params/*.npz")), "files", flush=True)

sh("pip install -q biopython")
import urllib.request
import numpy as np
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import protein_letters_3to1

os.makedirs("/kaggle/working/targets", exist_ok=True)
raw = "/kaggle/working/1TNF.pdb"
if not os.path.exists(raw):
    open(raw, "wb").write(urllib.request.urlopen(
        "https://files.rcsb.org/download/1TNF.pdb", timeout=180).read())

P = PDBParser(QUIET=True)
mdl = P.get_structure("m", raw)[0]
ca = {}
for ch in mdl:
    for r in ch:
        if r.id[0] == " " and "CA" in r:
            ca[(ch.id, r.id[1])] = r["CA"].get_coord()
centre = (ca[("A", 166 - OFF)] + ca[("B", 108 - OFF)]) / 2
keep = {k for k, v in ca.items() if k[0] in ("A", "B")
        and float(np.linalg.norm(v - centre)) <= RADIUS}


class Crop(Select):
    def accept_chain(self, c): return c.id in ("A", "B")
    def accept_residue(self, r): return r.id[0] == " " and (r.parent.id, r.id[1]) in keep
    def accept_atom(self, a): return a.element != "H"


TARGET = "/kaggle/working/targets/TNF_site_crop.pdb"
st = P.get_structure("t", raw)
io = PDBIO(); io.set_structure(st); io.save(TARGET, Crop())

# The map from construct order to the original chain and UniProt number. AfDesign
# CONCATENATES the target chains into one chain on output and renumbers from 1 - the
# smoke test showed chains {'A': 608, 'B': 140} where three chains were expected - so
# position in this list IS the output residue number minus one. Everything downstream
# maps through this rather than assuming a numbering.
order = []
for ch in P.get_structure("c", TARGET)[0]:
    for r in ch:
        if r.id[0] == " ":
            order.append(dict(chain=ch.id, pdb=r.id[1], uniprot=r.id[1] + OFF,
                              aa=protein_letters_3to1.get(r.get_resname(), "X")))
json.dump(dict(radius=RADIUS, n=len(order), order=order),
          open("/kaggle/working/construct_map.json", "w"))

SITE = [95, 105, 106, 107, 108, 109, 110, 111, 112, 155, 159, 161, 162, 163, 164, 165,
        166, 167, 168, 201, 203, 207, 220, 221, 223]
have = {(o["chain"], o["uniprot"]) for o in order}
print("cropped target: %d residues (A %d, B %d)"
      % (len(order), sum(1 for o in order if o["chain"] == "A"),
         sum(1 for o in order if o["chain"] == "B")), flush=True)
print("site residues retained: %d of %d"
      % (len({o["uniprot"] for o in order} & set(SITE)), len(SITE)), flush=True)
assert ("A", 166) in have and ("B", 108) in have, "a switch anchor was cropped away"
assert len({o["uniprot"] for o in order} & set(SITE)) == len(SITE), "site residue lost"
print("switch anchors present: K166(A), R108(B)", flush=True)

HOTSPOT = "A86,A87,A88,A90,A91,A92,B29,B32,B33,B34,B35,B36"

PROBE = ("python -c \"import jax, jax.lib; assert hasattr(jax.lib,'xla_bridge'); "
         "print(jax.__version__)\"")
r = subprocess.run(PROBE, shell=True, capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip().startswith("0.4.38"):
    print("installing jax 0.4.38 ...", flush=True)
    sh("pip install -q 'jax[cuda12]==0.4.38' 'jaxlib==0.4.38'")
sh("pip install -q git+https://github.com/sokrypton/ColabDesign.git")

chk = subprocess.run(
    "cd /kaggle/working && python -c \""
    "import jax; from colabdesign import mk_afdesign_model; "
    "m = mk_afdesign_model(protocol='binder', data_dir='/kaggle/working'); "
    "m.prep_inputs(pdb_filename='%s', chain='A,B', binder_len=65, hotspot='%s'); "
    "print('READY', jax.__version__, jax.devices()); "
    "print('target length seen by the model:', m._target_len)\"" % (TARGET, HOTSPOT),
    shell=True, capture_output=True, text=True)
print(chk.stdout[-500:])
if chk.returncode != 0:
    print("PREP FAILED:"); print(chk.stderr[-1500:])
print("TNF PRODUCTION SETUP DONE in %.1f min" % ((time.time() - t0) / 60))


In [ ]:
# === CHALLENGE 2 (TNF-alpha) : MOUSE TARGET SETUP ===
#
# Objective 2 is mouse cross-reactivity, and it is ranked above affinity. Until now it
# has only been inferred from sequence conservation across each design's human
# epitope. This builds the mouse equivalent of the human target so binding can be
# predicted directly instead.
#
# PDB 2TNF is mouse TNF-alpha at 1.4 A (UniProt P06804, chains A/B/C, a trimer) -
# higher resolution than the human 1TNF used for design.
#
# The numbering is the part that has to be right. Human 1TNF is numbered UniProt-76;
# mouse 2TNF is numbered UniProt-79. Worse, the human->mouse alignment is NOT a
# constant shift: across the 25 site residues the offset is +2 at some positions and
# +3 at others, so there is an indel between the orthologs. Every mouse residue here
# therefore comes from the pairwise alignment, never from arithmetic on a human
# number. On EGFR a three-residue numbering shift silently emptied a result list for
# two rounds, so each step below asserts what it believes.
#
# CORRECTED after the first attempt failed on its own assertion with
# "mouse 168 is GLU, expected LYS". 2TNF's author numbering is NOT a constant offset
# from UniProt: the chain runs 9-157, which is 149 positions, while its DBREF claims
# 1-156, and the observed offset takes BOTH the values 78 and 79 within a single
# chain. Mouse K168 is residue 90, not 89. Residue numbers are therefore derived by
# aligning each observed chain sequence to P06804 and are never computed by adding a
# constant - the same rule already applied to the human construct, which this cell
# originally failed to follow.
import glob, json, os, shutil, subprocess, sys, time
t0 = time.time()
RADIUS = 22.0                  # same sphere as the human crop

# UniProt P06804, mouse TNF-alpha precursor, 235 aa. Embedded so the mapping does not
# depend on a second network fetch at run time.
P06804 = (
    "MSTESMIRDVELAEEALPQKMGGFQNSRRCLCLSLFSFLLVAGATTLFCLLNFGVIGPQR"
    "DEKFPNGLPLISSMAQTLTLRSSSQNSSDKPVAHVVANHQVEEQLEWLSQRANALLANGM"
    "DLKDNQLVVPADGLYLVYSQVLFKGQGCPDYVLLTHTVSRFAISYQEKVNLLSAVKSPCP"
    "KDTPEGAELKPWYEPIYLGGVFQLEKGDQLSAEVNLPKYLDFAESGQVYFGVIAL")

# From the pairwise alignment of P01375 and P06804 (tnf/results/human_to_mouse_index.json).
# Mouse UniProt numbers for the 25 human site residues, in human order.
MOUSE_SITE = [98, 108, 109, 110, 111, 112, 113, 114, 115, 157, 161, 163, 164, 165,
              166, 167, 168, 169, 170, 203, 205, 209, 222, 223, 225]
HUMAN_SITE = [95, 105, 106, 107, 108, 109, 110, 111, 112, 155, 159, 161, 162, 163,
              164, 165, 166, 167, 168, 201, 203, 207, 220, 221, 223]
ANCHOR_K, ANCHOR_R = 168, 111  # mouse equivalents of human K166 and R108


def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("FAILED:", cmd)
        print((r.stdout + r.stderr)[-1000:])
        if check:
            raise SystemExit("setup failed")
    return r


print("python", sys.version.split()[0], flush=True)
print(subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip(), flush=True)

os.makedirs("/kaggle/working/params", exist_ok=True)
src = glob.glob("/kaggle/input/**/params/params_model_*.npz", recursive=True)
if src:
    print("reusing %d AF2 param files from the attached input" % len(src), flush=True)
    for f in src:
        dst = "/kaggle/working/params/" + os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy(f, dst)
else:
    print("downloading AF2 params (~3.5 GB) ...", flush=True)
    sh("wget -q -O /kaggle/working/af2.tar "
       "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar")
    sh("tar -xf /kaggle/working/af2.tar -C /kaggle/working/params")
    sh("rm -f /kaggle/working/af2.tar")
print("AF2 params present:", len(glob.glob("/kaggle/working/params/*.npz")), "files", flush=True)

sh("pip install -q biopython")
import urllib.request
import numpy as np
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import protein_letters_3to1

os.makedirs("/kaggle/working/targets", exist_ok=True)
raw = "/kaggle/working/2TNF.pdb"
if not os.path.exists(raw):
    open(raw, "wb").write(urllib.request.urlopen(
        "https://files.rcsb.org/download/2TNF.pdb", timeout=180).read())

from Bio import Align
_al = Align.PairwiseAligner()
_al.open_gap_score, _al.extend_gap_score = -11, -1
_al.substitution_matrix = Align.substitution_matrices.load("BLOSUM62")

P = PDBParser(QUIET=True)
mdl = P.get_structure("m", raw)[0]

# residue number -> UniProt index, per chain, by ALIGNMENT not arithmetic
R2U = {}
for cid in ("A", "B"):
    res = [r for r in mdl[cid] if r.id[0] == " "]
    seq = "".join(protein_letters_3to1.get(r.get_resname(), "X") for r in res)
    aln = _al.align(P06804, seq)[0]
    mp = {}
    for (us, ue), (ss, se) in zip(aln.aligned[0], aln.aligned[1]):
        for k in range(int(ue - us)):
            mp[res[int(ss + k)].id[1]] = int(us + k) + 1
    bad = [(n, u) for n, u in mp.items()
           if P06804[u - 1] != protein_letters_3to1.get(mdl[cid][n].get_resname(), "X")]
    assert not bad, "chain %s: %d residues disagree with P06804, e.g. %s" % (cid, len(bad), bad[:3])
    R2U[cid] = mp
    print("chain %s: %d residues mapped to P06804, 0 mismatches, offsets present %s"
          % (cid, len(mp), sorted({u - n for n, u in mp.items()})), flush=True)

U2R = {cid: {u: n for n, u in mp.items()} for cid, mp in R2U.items()}
ca, name3 = {}, {}
for ch in mdl:
    for r in ch:
        if r.id[0] == " " and "CA" in r:
            ca[(ch.id, r.id[1])] = r["CA"].get_coord()
            name3[(ch.id, r.id[1])] = r.get_resname()

assert ANCHOR_K in U2R["A"], "mouse K%d is absent from 2TNF chain A" % ANCHOR_K
assert ANCHOR_R in U2R["B"], "mouse R%d is absent from 2TNF chain B" % ANCHOR_R
kA, rB = ("A", U2R["A"][ANCHOR_K]), ("B", U2R["B"][ANCHOR_R])
assert name3[kA] == "LYS", "mouse %d (resi A%d) is %s, expected LYS" % (ANCHOR_K, kA[1], name3[kA])
assert name3[rB] == "ARG", "mouse %d (resi B%d) is %s, expected ARG" % (ANCHOR_R, rB[1], name3[rB])
print("anchors confirmed in 2TNF: K%d = resi A%d, R%d = resi B%d"
      % (ANCHOR_K, kA[1], ANCHOR_R, rB[1]), flush=True)

centre = (ca[kA] + ca[rB]) / 2
keep = {k for k, v in ca.items() if k[0] in ("A", "B")
        and float(np.linalg.norm(v - centre)) <= RADIUS}


class Crop(Select):
    def accept_chain(self, c): return c.id in ("A", "B")
    def accept_residue(self, r): return r.id[0] == " " and (r.parent.id, r.id[1]) in keep
    def accept_atom(self, a): return a.element != "H"


TARGET = "/kaggle/working/targets/TNF_mouse_site_crop.pdb"
st = P.get_structure("t", raw)
io = PDBIO(); io.set_structure(st); io.save(TARGET, Crop())

order = []
for ch in P.get_structure("c", TARGET)[0]:
    for r in ch:
        if r.id[0] == " ":
            order.append(dict(chain=ch.id, pdb=r.id[1], uniprot=R2U[ch.id][r.id[1]],
                              aa=protein_letters_3to1.get(r.get_resname(), "X")))
H2M = dict(zip(HUMAN_SITE, MOUSE_SITE))
M2H = {m: h for h, m in H2M.items()}
json.dump(dict(species="mouse", pdb="2TNF", radius=RADIUS, n=len(order),
               numbering="per-residue, from alignment to P06804; offset is not constant",
               order=order, mouse_site=MOUSE_SITE, human_site=HUMAN_SITE,
               mouse_to_human_site=M2H),
          open("/kaggle/working/construct_map_mouse.json", "w"))

have = {o["uniprot"] for o in order}
print("cropped mouse target: %d residues (A %d, B %d)"
      % (len(order), sum(1 for o in order if o["chain"] == "A"),
         sum(1 for o in order if o["chain"] == "B")), flush=True)
print("mouse site residues retained: %d of %d" % (len(have & set(MOUSE_SITE)),
                                                  len(MOUSE_SITE)), flush=True)
missing = sorted(set(MOUSE_SITE) - have)
assert not missing, "mouse site residues cropped away: %s" % missing
assert ("A", ANCHOR_K) in {(o["chain"], o["uniprot"]) for o in order}
assert ("B", ANCHOR_R) in {(o["chain"], o["uniprot"]) for o in order}
print("switch anchors present: K%d(A), R%d(B)" % (ANCHOR_K, ANCHOR_R), flush=True)

# Sanity check the alignment itself: every site residue identity must match what the
# human/mouse alignment predicted, or the mapping is wrong somewhere.
seqmap = {o["uniprot"]: o["aa"] for o in order}
HUMAN_AA = dict(zip(HUMAN_SITE, "NLNRRANALTIVSYQTKVNQERFAS"))
bad = [(m, seqmap[m]) for m in MOUSE_SITE if m in seqmap and seqmap[m] not in "ACDEFGHIKLMNPQRSTVWY"]
assert not bad, "unparsable residues %s" % bad
print("mouse site sequence: %s" % "".join(seqmap[m] for m in MOUSE_SITE), flush=True)
print("human site sequence: %s" % "".join(HUMAN_AA[h] for h in HUMAN_SITE), flush=True)

HOTSPOT = ",".join(
    sorted({"%s%d" % (o["chain"], o["pdb"]) for o in order
            if o["uniprot"] in set(MOUSE_SITE)}))
open("/kaggle/working/mouse_hotspot.txt", "w").write(HOTSPOT)
print("mouse hotspot string: %s" % HOTSPOT, flush=True)

PROBE = ("python -c \"import jax, jax.lib; assert hasattr(jax.lib,'xla_bridge'); "
         "print(jax.__version__)\"")
r = subprocess.run(PROBE, shell=True, capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip().startswith("0.4.38"):
    print("installing jax 0.4.38 ...", flush=True)
    sh("pip install -q 'jax[cuda12]==0.4.38' 'jaxlib==0.4.38'")
sh("pip install -q git+https://github.com/sokrypton/ColabDesign.git")

chk = subprocess.run(
    "cd /kaggle/working && python -c \""
    "import jax; from colabdesign import mk_afdesign_model; "
    "m = mk_afdesign_model(protocol='binder', data_dir='/kaggle/working'); "
    "m.prep_inputs(pdb_filename='%s', chain='A,B', binder_len=65); "
    "print('READY', jax.__version__, jax.devices()); "
    "print('mouse target length seen by the model:', m._target_len)\"" % TARGET,
    shell=True, capture_output=True, text=True)
print(chk.stdout[-500:])
if chk.returncode != 0:
    print("PREP FAILED:"); print(chk.stderr[-1500:])
print("MOUSE TARGET SETUP DONE in %.1f min" % ((time.time() - t0) / 60))


In [ ]:
# === CHALLENGE 2 (TNF-alpha) : EVIDENCE FOR THE SWITCH, AND ERROR BARS ===
#
# Everything submitted so far rests on binding at pH 7.4 against two species. Objective 1
# - the switch - has never been computed. No calculation in this project touches
# protonation. This run produces the first evidence about it, plus the error bars the
# ranking has been missing, plus an answer to a question the Proteinbase portal raised.
#
# Four measurements, on the 20 designs actually submitted (re-derived here from the
# switch5 output by the same rule, so no sequence is ever written into the repo):
#
# 1. CHARGE SENSITIVITY. The switch claim is that a protonated histidine cannot be
#    buried against a target cation. AlphaFold2 knows nothing about pH, but it does
#    respond to charge. So at each histidine position, predict Gln (neutral, similar
#    size) and Lys (charged, similar length). The DIFFERENCE isolates charge from
#    sterics. If Lys collapses the interface and Gln does not, the position genuinely
#    punishes positive charge and the switch should fire. If Lys is tolerated, our
#    switch argument for that design is unsupported. AF2 is a poor dG predictor, so
#    only large effects are interpretable - but this is falsifiable, which is new.
#
# 2. MODEL-ENSEMBLE ERROR BARS. Every i_ptm reported so far is a single prediction from
#    one parameter set. Across-model spread is typically +/-0.05 to 0.10, the same size
#    as several margins the ranking depends on. Five models, mean and sd.
#
# 3. MONOMER FOLDABILITY. The portal reported pLDDT 55-63 where our complex predictions
#    gave 60-77. The likely explanation is that it predicted the binder ALONE. If these
#    are target-dependent folders they are poor bets to express. Predicted here directly
#    rather than inferred.
#
# 4. BURIAL ON BINDING. The proposal geometry used Cbeta-to-cation distance with a
#    3.0-8.8 A window. That is the wrong variable. The switch is a binding-coupled
#    protonation: in the free binder the histidine is exposed and protonates at pH 6;
#    binding has to bury that charge, and the burial is what costs affinity. So measure
#    histidine side-chain rel_SASA free vs bound, the real Nd1/Ne2-to-cation distance
#    after folding, and a Debye-screened Coulomb estimate at 150 mM. CPU only.
#
# Nothing here redesigns anything. No sequence is changed except the deliberate
# Gln/Lys probes, which are never submitted.
import glob, json, os, subprocess, sys, textwrap, time

TIME_BUDGET_MIN = 210
N_SWITCH = 12
TOTAL_SLOTS = 20
N_MODELS = 5

WORKER = r"""
import glob, json, os, sys, time
from collections import Counter
GPU, SHARD, BUDGET = sys.argv[1], int(sys.argv[2]), float(sys.argv[3])
N_SWITCH, TOTAL_SLOTS, N_MODELS = int(sys.argv[4]), int(sys.argv[5]), int(sys.argv[6])
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
t0 = time.time()
def log(m): print("[w%d] %s [%.1f min]" % (SHARD, m, (time.time()-t0)/60), flush=True)

import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley
from Bio.PDB.Polypeptide import protein_letters_3to1

HT = "/kaggle/working/targets/TNF_site_crop.pdb"
MT = "/kaggle/working/targets/TNF_mouse_site_crop.pdb"
CMAP = json.load(open("/kaggle/working/construct_map.json"))["order"]
H_HOTSPOT = "A86,A87,A88,A90,A91,A92,B29,B32,B33,B34,B35,B36"
M_HOTSPOT = open("/kaggle/working/mouse_hotspot.txt").read().strip()
CAT_N = {"ARG": ("NE", "NH1", "NH2"), "LYS": ("NZ",)}
MAXS = {"A":129,"R":274,"N":195,"D":193,"C":167,"E":223,"Q":225,"G":104,"H":224,
        "I":197,"L":201,"K":236,"M":224,"F":240,"P":159,"S":155,"T":172,"W":285,
        "Y":263,"V":174}
# Debye-Huckel at 150 mM monovalent, 298 K: kappa^-1 = 7.9 A. eps_eff = 30 is the
# usual compromise for a partially desolvated protein-protein interface; the absolute
# number is soft, the RANKING across positions is what this is for.
EPS_EFF, DEBYE = 30.0, 7.9

# ---- rebuild the submitted set from the switch5 output, by the same rule -----------
src = glob.glob("/kaggle/input/**/tnf_switch5.json", recursive=True)
if not src:
    raise SystemExit("tnf_switch5.json not found - attach the tnf_switch5_repair_fix "
                     "output. Do NOT read Kaggle's file PREVIEW for this: it truncates "
                     "at 102400 bytes and closes the JSON brackets, so a short read "
                     "parses as valid and silently drops records.")
recs = json.load(open(src[0]))
if len(recs) < 26:
    raise SystemExit("only %d records in tnf_switch5.json - expected 29; the file looks "
                     "truncated, refusing to select from it" % len(recs))

def rank_key(v):
    return (v["n_his"], v["n_anchor"], min(v["human_i_ptm"], v["mouse_i_ptm"]))

chosen = []
for r in recs:
    ok = [v for v in r.get("variants", []) if v.get("passes")]
    if not ok:
        continue
    best = max(ok, key=rank_key)
    best = dict(best); best["parent"] = r["name"]; best["control"] = r["control"]
    chosen.append(best)
chosen.sort(key=rank_key, reverse=True)
picked = chosen[:N_SWITCH]
n_ctl = TOTAL_SLOTS - len(picked)
clean = [v for v in picked if "H" not in v["control"]["sequence"]]
with_ctl = {v["parent"] for v in clean[:n_ctl]}
if SHARD == 0:
    log("%d lineages usable; %d switches, %d controls" % (len(chosen), len(picked), n_ctl))

jobs = []
for i, v in enumerate(picked, 1):
    jobs.append(dict(slot="sw_%02d" % i, role="switch", parent=v["parent"],
                     seq=v["sequence"], his=v["his_substitutions"],
                     ctl_seq=v["control"]["sequence"], ref_h=v["human_i_ptm"],
                     ref_m=v["mouse_i_ptm"]))
    if v["parent"] in with_ctl:
        jobs.append(dict(slot="ct_%02d" % i, role="control", parent=v["parent"],
                         seq=v["control"]["sequence"], his="",
                         ctl_seq=v["control"]["sequence"],
                         ref_h=v["control"]["human_i_ptm"],
                         ref_m=v["control"]["mouse_i_ptm"]))
# Sorting by length so each model rebuild is amortised over every job of that length.
jobs.sort(key=lambda j: (len(j["seq"]), j["slot"]))
mine = [j for k, j in enumerate(jobs) if k % 2 == SHARD]
log("this worker takes %d of %d designs: %s"
    % (len(mine), len(jobs), ", ".join(j["slot"] for j in mine)))

from colabdesign import mk_afdesign_model, clear_mem
_c = {}

def _model(species, L):
    key = (species, L)
    if _c.get("key") != key:
        clear_mem()
        if species == "alone":
            af = mk_afdesign_model(protocol="hallucination", length=L,
                                   data_dir="/kaggle/working")
        else:
            af = mk_afdesign_model(protocol="binder", data_dir="/kaggle/working")
            af.prep_inputs(pdb_filename=HT if species == "human" else MT, chain="A,B",
                           binder_len=L,
                           hotspot=H_HOTSPOT if species == "human" else M_HOTSPOT)
        _c["key"], _c["af"] = key, af
    return _c["af"]

def predict(seq, species, out_pdb=None, model=None):
    af = _model(species, len(seq))
    kw = dict(seq=seq, num_recycles=3, verbose=False)
    if model is not None:
        kw["models"] = [model]
    try:
        af.predict(**kw)
    except TypeError:
        kw.pop("models", None)
        kw.pop("num_recycles", None)
        af.predict(**kw)
        if model is not None:
            return None            # per-model selection unavailable in this build
    lg = af.aux["log"]
    if out_pdb:
        af.save_pdb(out_pdb)
    return (round(float(lg.get("i_ptm", lg.get("ptm", 0.0))), 3),
            round(float(lg.get("plddt", 0.0)), 3))

# ---- burial on binding, from the complex PDB --------------------------------------
# A chain written twice overlaps itself exactly, so every one of its residues reports
# as fully buried and the whole measurement silently inverts. This project has already
# been bitten by duplicated chains in a saved PDB, so every chain is de-duplicated
# BEFORE any SASA is computed, not after.
def dedupe(chain):
    res, seen = [], set()
    for r in list(chain):
        if r.id[0] != " ":
            chain.detach_child(r.id); continue
        if r.id[1] in seen:
            chain.detach_child(r.id); continue
        seen.add(r.id[1]); res.append(r)
    return res

def burial(pdb_path, blen, his_idx):
    # rel_SASA of each histidine side chain in the complex vs in the binder alone,
    # the real Nd1/Ne2-to-cation distance after folding, and a screened Coulomb term.
    mdl = PDBParser(QUIET=True).get_structure("c", pdb_path)[0]
    kept = {ch.id: dedupe(ch) for ch in mdl}
    bnd_id = min(kept, key=lambda k: len(kept[k]))
    bres = kept[bnd_id]
    if len(bres) != blen:
        raise ValueError("binder chain %s has %d unique residues, expected %d"
                         % (bnd_id, len(bres), blen))
    # cationic nitrogens on the target, from the complex as folded
    cats = []
    for cid, rs in kept.items():
        if cid == bnd_id:
            continue
        for r in rs:
            for an in CAT_N.get(r.get_resname(), ()):
                if an in r:
                    cats.append((cid, r.id[1], an, r[an].get_coord()))
    if not cats:
        raise ValueError("no Arg/Lys nitrogens found on the target chains")
    # SASA twice: whole de-duplicated complex, then the binder chain on its own
    ShrakeRupley().compute(mdl, level="R")
    bound = {}
    for k, r in enumerate(bres):
        aa = protein_letters_3to1.get(r.get_resname())
        if aa:
            bound[k] = r.sasa / MAXS.get(aa, 200)
    import copy
    solo = copy.deepcopy(mdl[bnd_id])
    sres = [r for r in solo if r.id[0] == " "]
    ShrakeRupley().compute(solo, level="R")
    free = {}
    for k, r in enumerate(sres):
        aa = protein_letters_3to1.get(r.get_resname())
        if aa:
            free[k] = r.sasa / MAXS.get(aa, 200)
    out = {}
    for k in his_idx:
        r = bres[k]
        best = None
        for an in ("ND1", "NE2"):
            if an not in r:
                continue
            for cid, cnum, catn, xyz in cats:
                d = float(np.linalg.norm(r[an].get_coord() - xyz))
                if best is None or d < best[0]:
                    best = (d, "%s%d%s" % (cid, cnum, catn), an)
        ddg = None
        if best is not None:
            # 332 * q1*q2 / (eps * r) * exp(-r/lambda), kcal/mol, both charges +1
            ddg = round(332.0 / (EPS_EFF * best[0]) * np.exp(-best[0] / DEBYE), 3)
        out[k] = dict(rel_sasa_free=round(free.get(k, -1), 3),
                      rel_sasa_bound=round(bound.get(k, -1), 3),
                      buried_on_binding=round(free.get(k, 0) - bound.get(k, 0), 3),
                      nearest_cation=best[1] if best else None,
                      cation_dist=round(best[0], 2) if best else None,
                      which_n=best[2] if best else None,
                      screened_coulomb_kcal=ddg)
    return out

os.makedirs("/kaggle/working/ev_pdb", exist_ok=True)
rec_path = "/kaggle/working/tnf_evidence_w%d.jsonl" % SHARD

def emit(d):
    with open(rec_path, "a") as fh:
        fh.write(json.dumps(d) + chr(10))

for j in mine:
    if (time.time() - t0) / 60 > BUDGET:
        log("%s not started: time budget reached" % j["slot"])
        emit(dict(slot=j["slot"], role=j["role"], parent=j["parent"],
                  incomplete="time budget reached before this design"))
        continue
    seq, L = j["seq"], len(j["seq"])
    rec = dict(slot=j["slot"], role=j["role"], parent=j["parent"], length=L,
               his=j["his"], ref_human_i_ptm=j["ref_h"], ref_mouse_i_ptm=j["ref_m"])
    try:
        # --- 2. error bars, human then mouse
        for species in ("human", "mouse"):
            vals = []
            for k in range(N_MODELS):
                out = "/kaggle/working/ev_pdb/%s_%s_m%d.pdb" % (j["slot"], species, k)
                r = predict(seq, species, out_pdb=out if k == 0 else None, model=k)
                if r is None:
                    rec["%s_models" % species] = "per-model selection unavailable"
                    break
                vals.append(r)
            if vals:
                ip = [v[0] for v in vals]
                rec["%s_i_ptm_mean" % species] = round(float(np.mean(ip)), 3)
                rec["%s_i_ptm_sd" % species] = round(float(np.std(ip)), 3)
                rec["%s_i_ptm_all" % species] = ip
                rec["%s_plddt_mean" % species] = round(float(np.mean([v[1] for v in vals])), 3)
        log("%s human %.3f+/-%.3f  mouse %.3f+/-%.3f (ref %.3f/%.3f)"
            % (j["slot"], rec.get("human_i_ptm_mean", 0), rec.get("human_i_ptm_sd", 0),
               rec.get("mouse_i_ptm_mean", 0), rec.get("mouse_i_ptm_sd", 0),
               j["ref_h"], j["ref_m"]))

        # --- 3. does it fold on its own
        try:
            alone = predict(seq, "alone")
            if alone:
                rec["alone_plddt"] = alone[1]
                rec["plddt_drop_alone"] = round(rec.get("human_plddt_mean", 0) - alone[1], 3)
                log("%s alone pLDDT %.3f vs %.3f in complex (drop %.3f)"
                    % (j["slot"], alone[1], rec.get("human_plddt_mean", 0),
                       rec["plddt_drop_alone"]))
        except Exception as e:
            rec["alone_error"] = "%s %s" % (type(e).__name__, str(e)[:120])

        # --- 1. charge sensitivity, and 4. burial - switches only
        if j["role"] == "switch" and j["his"]:
            hidx = [int(s[1:-1]) - 1 for s in j["his"].split(",")]
            probes = []
            for pos in hidx:
                assert seq[pos] == "H", "%s expected H at %d" % (j["slot"], pos + 1)
                for aa, label in (("Q", "neutral"), ("K", "charged")):
                    mut = seq[:pos] + aa + seq[pos+1:]
                    hp = predict(mut, "human")
                    mp = predict(mut, "mouse")
                    probes.append(dict(position=pos+1, residue=aa, kind=label,
                                       human_i_ptm=hp[0], mouse_i_ptm=mp[0]))
            rec["charge_probes"] = probes
            for pos in hidx:
                q = next((p for p in probes if p["position"] == pos+1 and p["residue"] == "Q"), None)
                kk = next((p for p in probes if p["position"] == pos+1 and p["residue"] == "K"), None)
                if q and kk:
                    d = round(kk["human_i_ptm"] - q["human_i_ptm"], 3)
                    rec.setdefault("charge_sensitivity", {})["H%d" % (pos+1)] = d
                    log("%s H%d charge test: Gln %.3f, Lys %.3f -> %+.3f %s"
                        % (j["slot"], pos+1, q["human_i_ptm"], kk["human_i_ptm"], d,
                           "punishes positive charge" if d <= -0.10
                           else "tolerates positive charge - switch unsupported"))
            try:
                pdb0 = "/kaggle/working/ev_pdb/%s_human_m0.pdb" % j["slot"]
                rec["burial"] = {("H%d" % (k+1)): v
                                 for k, v in burial(pdb0, L, hidx).items()}
                for k, v in rec["burial"].items():
                    log("%s %s burial: free %.3f -> bound %.3f (buried %.3f), "
                        "nearest cation %s at %.2f A, screened %.3f kcal/mol"
                        % (j["slot"], k, v["rel_sasa_free"], v["rel_sasa_bound"],
                           v["buried_on_binding"], v["nearest_cation"],
                           v["cation_dist"] or -1, v["screened_coulomb_kcal"] or 0))
            except Exception as e:
                rec["burial_error"] = "%s %s" % (type(e).__name__, str(e)[:160])
    except Exception as e:
        rec["error"] = "%s %s" % (type(e).__name__, str(e)[:200])
        log("%s FAILED %s" % (j["slot"], rec["error"]))
    emit(rec)
log("worker done")
"""

with open("/kaggle/working/tnf_evidence_worker.py", "w") as fh:
    fh.write(textwrap.dedent(WORKER))

subprocess.run("pip install -q biopython", shell=True)
NEEDED = ["/kaggle/working/targets/TNF_site_crop.pdb",
          "/kaggle/working/targets/TNF_mouse_site_crop.pdb",
          "/kaggle/working/construct_map.json",
          "/kaggle/working/construct_map_mouse.json",
          "/kaggle/working/mouse_hotspot.txt"]
missing = [p for p in NEEDED if not os.path.exists(p)]
if missing:
    raise SystemExit("setup cells have not run - missing %s" % missing)

procs = []
for shard, gpu in enumerate(["0", "1"]):
    p = subprocess.Popen([sys.executable, "-u", "/kaggle/working/tnf_evidence_worker.py",
                          gpu, str(shard), str(TIME_BUDGET_MIN), str(N_SWITCH),
                          str(TOTAL_SLOTS), str(N_MODELS)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    procs.append(p)
    print("launched evidence worker %d on GPU %s" % (shard, gpu), flush=True)

import threading
def pump(p):
    for line in p.stdout:
        print(line.rstrip(), flush=True)
threads = [threading.Thread(target=pump, args=(p,), daemon=True) for p in procs]
[t.start() for t in threads]
for p in procs:
    p.wait()
[t.join(timeout=10) for t in threads]

rows = []
for f in sorted(glob.glob("/kaggle/working/tnf_evidence_w*.jsonl")):
    for line in open(f):
        line = line.strip()
        if line:
            rows.append(json.loads(line))
rows.sort(key=lambda r: r.get("slot", ""))
json.dump(rows, open("/kaggle/working/tnf_evidence.json", "w"), indent=1)
if not rows:
    raise SystemExit("nothing was measured - read the worker log above, do not read an "
                     "empty table as 'the switch does not work'")

print("\n==== ERROR BARS ON THE SUBMITTED SET ====")
print("%-6s %-8s %14s %14s %8s %8s"
      % ("slot", "role", "human mean+/-sd", "mouse mean+/-sd", "alone", "drop"))
for r in rows:
    print("%-6s %-8s %8.3f+/-%.3f %8.3f+/-%.3f %8s %8s"
          % (r.get("slot"), r.get("role", ""), r.get("human_i_ptm_mean", 0),
             r.get("human_i_ptm_sd", 0), r.get("mouse_i_ptm_mean", 0),
             r.get("mouse_i_ptm_sd", 0),
             ("%.3f" % r["alone_plddt"]) if "alone_plddt" in r else "-",
             ("%+.3f" % r["plddt_drop_alone"]) if "plddt_drop_alone" in r else "-"))

print("\n==== CHARGE SENSITIVITY: does the position punish a positive charge? ====")
print("A negative number means Lys costs more than Gln at that position, i.e. the")
print("position rejects positive charge and the switch has a reason to fire.")
print("%-6s %-14s %6s %8s %8s %8s  %s"
      % ("slot", "lineage", "his", "Gln", "Lys", "Lys-Gln", "reading"))
supported, unsupported = [], []
for r in rows:
    for key, d in (r.get("charge_sensitivity") or {}).items():
        pos = int(key[1:])
        pr = {p["residue"]: p for p in r.get("charge_probes", []) if p["position"] == pos}
        good = d <= -0.10
        (supported if good else unsupported).append((r["slot"], key, d))
        print("%-6s %-14s %6s %8.3f %8.3f %+8.3f  %s"
              % (r["slot"], r.get("parent", ""), key,
                 pr.get("Q", {}).get("human_i_ptm", 0),
                 pr.get("K", {}).get("human_i_ptm", 0), d,
                 "punishes +" if good else "tolerates + : UNSUPPORTED"))
print("\n%d histidine positions punish a positive charge; %d tolerate it"
      % (len(supported), len(unsupported)))
if unsupported:
    print("switch claim NOT supported at: %s"
          % ", ".join("%s %s" % (s, k) for s, k, _ in unsupported))

print("\n==== BURIAL ON BINDING (the variable we should have filtered on) ====")
print("%-6s %6s %8s %8s %8s %12s %7s %9s"
      % ("slot", "his", "free", "bound", "buried", "cation", "dist", "kcal/mol"))
for r in rows:
    for key, v in (r.get("burial") or {}).items():
        print("%-6s %6s %8.3f %8.3f %8.3f %12s %7s %9s"
              % (r["slot"], key, v["rel_sasa_free"], v["rel_sasa_bound"],
                 v["buried_on_binding"], v["nearest_cation"] or "-",
                 v["cation_dist"] or "-", v["screened_coulomb_kcal"] or "-"))

print("\n===== COPY FROM HERE =====")
for r in rows:
    print(json.dumps(r, separators=(",", ":")))
print("===== COPY TO HERE =====")
